In [1]:
import pandas as pd

In [2]:
hunters = pd.read_csv("../data/hunters.csv")
gatherers = pd.read_csv("../data/gatherers.csv")
all_participants = pd.concat([hunters, gatherers], ignore_index=True)

In [3]:
gath_fold_0 = pd.read_csv("../data/GatherersFolds/fold_0_trial_ids_by_regime.csv")

In [40]:
mask = ~gath_fold_0["unique_trial_id"].astype(str).str.endswith("_0_0")

print("Any non _0_0 endings:", mask.any())

Any non _0_0 endings: False


In [4]:
hunt_all_folds = pd.read_csv("../data/HuntingIsCorrectFolds/all_folds_subjects_items.csv")

In [26]:
gath_fold_0['regime'].value_counts()

regime
train_train                        5855
test_seen_subject_unseen_item      1080
val_seen_subject_unseen_item        960
test_unseen_subject_seen_item       851
val_unseen_subject_seen_item        732
val_unseen_subject_unseen_item      120
test_unseen_subject_unseen_item     120
Name: count, dtype: int64

In [18]:
hunt_all_folds

,eval_regime,eval_type,fold,participant_id,article_batch,article_id
0,train,train,0,l10_39,1,9
1,train,train,0,l10_39,1,2
2,train,train,0,l10_39,1,3
3,train,train,0,l10_39,1,5
4,train,train,0,l10_39,1,7
...,...,...,...,...,...,...
17995,new_subject,val,9,l7_183,2,3
17996,new_subject,val,9,l7_183,2,6
17997,new_subject,val,9,l7_183,2,10
17998,new_subject,val,9,l7_183,2,4


In [10]:
one_for_test = hunt_all_folds[hunt_all_folds['participant_id'] == 'l10_39']
one_for_test

,eval_regime,eval_type,fold,participant_id,article_batch,article_id
0,train,train,0,l10_39,1,9
1,train,train,0,l10_39,1,2
2,train,train,0,l10_39,1,3
3,train,train,0,l10_39,1,5
4,train,train,0,l10_39,1,7
...,...,...,...,...,...,...
16349,train,train,9,l10_39,1,7
16350,train,train,9,l10_39,1,6
16351,train,train,9,l10_39,1,4
17550,new_item,test,9,l10_39,1,1


In [24]:
one_for_test['eval_regime'].value_counts()

eval_regime
train                   64
new_item                17
new_subject             17
new_item_and_subject     2
Name: count, dtype: int64

In [8]:
hunters[['participant_id','article_batch', 'article_id']]

,participant_id,article_batch,article_id
0,l42_2070,3,6
1,l42_2070,3,6
2,l42_2070,3,6
3,l42_2070,3,6
4,l42_2070,3,6
...,...,...,...
380345,l10_39,1,5
380346,l10_39,1,5
380347,l10_39,1,5
380348,l10_39,1,5


In [14]:
one_fold_h = hunters.merge(one_for_test, on=['participant_id', 'article_batch', 'article_id'], how='left')
one_fold_h[['text_id', 'text_id_with_q', 'participant_id', 'article_batch', 'article_id']]

,text_id,text_id_with_q,participant_id,article_batch,article_id
0,3_6_Ele_1,3_6_Ele_1_2,l42_2070,3,6
1,3_6_Ele_1,3_6_Ele_1_2,l42_2070,3,6
2,3_6_Ele_1,3_6_Ele_1_2,l42_2070,3,6
3,3_6_Ele_1,3_6_Ele_1_2,l42_2070,3,6
4,3_6_Ele_1,3_6_Ele_1_2,l42_2070,3,6
...,...,...,...,...,...
401954,1_5_Adv_5,1_5_Adv_5_1,l10_39,1,5
401955,1_5_Adv_5,1_5_Adv_5_1,l10_39,1,5
401956,1_5_Adv_5,1_5_Adv_5_1,l10_39,1,5
401957,1_5_Adv_5,1_5_Adv_5_1,l10_39,1,5


In [19]:
hunters_join_cols = [
    "participant_id",
    "article_batch",
    "article_id",
    "text_id",                  # becomes unique_paragraph_id
    "TRIAL_INDEX",              # becoms unique_trial_id
    "list_number",
    "question_preview",
    "repeated_reading_trial",
    "is_correct",
]


hunters[hunters_join_cols]

,participant_id,article_batch,article_id,text_id,TRIAL_INDEX,list_number,question_preview,repeated_reading_trial,is_correct
0,l42_2070,3,6,3_6_Ele_1,4,42,True,False,1
1,l42_2070,3,6,3_6_Ele_1,4,42,True,False,1
2,l42_2070,3,6,3_6_Ele_1,4,42,True,False,1
3,l42_2070,3,6,3_6_Ele_1,4,42,True,False,1
4,l42_2070,3,6,3_6_Ele_1,4,42,True,False,1
...,...,...,...,...,...,...,...,...,...
380345,l10_39,1,5,1_5_Adv_5,59,10,True,False,1
380346,l10_39,1,5,1_5_Adv_5,59,10,True,False,1
380347,l10_39,1,5,1_5_Adv_5,59,10,True,False,1
380348,l10_39,1,5,1_5_Adv_5,59,10,True,False,1


In [35]:
import pandas as pd
from pathlib import Path

In [36]:
# ---------------------------
# paths
# ---------------------------
hunters_path = "../data/hunters.csv"
hunter_folds_path = "../data/HuntingIsCorrectFolds/all_folds_subjects_items.csv"
output_dir = Path("../data/HuntersFolds")
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------
# load
# ---------------------------
hunters = pd.read_csv(hunters_path)
hunt_all_folds = pd.read_csv(hunter_folds_path)

In [41]:

# ---------------------------
# build gatherer-like regime column
# ---------------------------
folds_df = hunt_all_folds.copy()

regime_map = {
    "train": "train",
    "new_item": "seen_subject_unseen_item",
    "new_subject": "unseen_subject_seen_item",
    "new_item_and_subject": "unseen_subject_unseen_item",
}

folds_df["regime"] = (
    folds_df["eval_type"].astype(str)
    + "_"
    + folds_df["eval_regime"].map(regime_map)
)

# ---------------------------
# join metadata from hunters
# ---------------------------
# keep only columns we actually want to bring in
hunters_join_cols = [
    "participant_id",
    "article_batch",
    "article_id",
    "text_id",                  # becomes unique_paragraph_id
    "list_number",
    "question_preview",
    "repeated_reading_trial",
    "is_correct",
]

hunters_meta = hunters[hunters_join_cols].drop_duplicates()

merged = folds_df.merge(
    hunters_meta,
    on=["participant_id", "article_batch", "article_id"],
    how="left",
)

# ---------------------------
# align column names to gatherer fold format
# ---------------------------
merged = merged.rename(columns={"text_id": "unique_paragraph_id"})

merged["unique_trial_id"] = (
    merged["participant_id"].astype(str)
    + "_"
    + merged["unique_paragraph_id"].astype(str)
    + "_1_0"
)

# desired output column order
target_cols = [
    "unique_paragraph_id",
    "participant_id",
    "unique_trial_id",
    "list_number",
    "question_preview",
    "repeated_reading_trial",
    "is_correct",
    "article_batch",
    "article_id",
    "regime",
]

# ---------------------------
# optional: quick diagnostics
# ---------------------------
print("Merged shape:", merged.shape)
print("\nMissing values per target column:")
print(merged[target_cols].isna().sum())


# ---------------------------
# cleanup: convert repeated_reading_trial to binary 0/1
# ---------------------------
merged["repeated_reading_trial"] = (
    merged["repeated_reading_trial"]
    .astype(str)
    .str.lower()
    .map({"true": 1, "false": 0})
)



merged["question_preview"] = merged["question_preview"].astype(str)

# ---------------------------
# write one file per fold
# ---------------------------
for fold_num in sorted(merged["fold"].dropna().unique()):
    fold_df = merged.loc[merged["fold"] == fold_num, target_cols].copy()

    out_path = output_dir / f"fold_{int(fold_num)}_trial_ids_by_regime.csv"
    fold_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({len(fold_df)} rows)")

Merged shape: (97190, 13)

Missing values per target column:
unique_paragraph_id       0
participant_id            0
unique_trial_id           0
list_number               0
question_preview          0
repeated_reading_trial    0
is_correct                0
article_batch             0
article_id                0
regime                    0
dtype: int64
Saved: ..\data\HuntersFolds\fold_0_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_1_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_2_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_3_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_4_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_5_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_6_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_7_trial_ids_by_regime.csv  (9719 rows)
Saved: ..\data\HuntersFolds\fold_8_trial_ids_by_regime.csv  (9719 rows